# 🧩 Day 3: Prompt Templates, Memory, Evaluation & Agent Sneak Peek — BrightPage Stationery & Office Supplies

This notebook covers Day 3 using **Google Gemini** (`gemini-3.5-flash-lite`) for **BrightPage Stationery & Office Supplies**:
1. **Setup** (`langchain-google-genai`, `.env` / Colab Secrets).
2. **LCEL Chain & Memory** (`ChatPromptTemplate` + `MessagesPlaceholder` + `StrOutputParser`).
3. **Automated Prompt Evaluation** (`tests.csv` — 10 BrightPage customer tests, `V1` -> `V2` comparison).
4. **Temperature & Stability** (`0.3` vs two runs at `1.0`).
5. **LangChain Agent Sneak Peek (`create_agent`)** & choosing between `plain code`, `one LLM call`, and `agent`.

In [1]:
!pip install -q --disable-pip-version-check langchain langchain-google-genai python-dotenv

In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

# 1. Load local .env if present
load_dotenv(find_dotenv())

# 2. Load from Google Colab Secrets if running in Colab
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = (
            userdata.get("GOOGLE_API_KEY") or userdata.get("GEMINI_API_KEY")
        )
        print("✅ Loaded GOOGLE_API_KEY from Google Colab Secrets.")
    except Exception:
        pass

# 3. Interactive fallback if still missing
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google Gemini API key: ")

import warnings
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings(
    "ignore",
    message=r"Model 'gemini-3\.5-flash-lite' uses fixed sampling defaults.*",
)

MODEL_NAME = "gemini-3.5-flash-lite"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0.3, max_retries=3, timeout=30)
print(f"🚀 Gemini LLM ready: {MODEL_NAME} (temperature=0.3)")

import time

def invoke_retry(runnable, payload, retries: int = 5):
    """Invoke a LangChain runnable with rate-limit-aware backoff (Gemini free tier: 15 RPM)."""
    for attempt in range(retries):
        try:
            return runnable.invoke(payload)
        except (KeyError, ValueError, FileNotFoundError):
            raise
        except Exception as err:
            if attempt == retries - 1:
                raise
            wait_s = 35 if "429" in str(err) or "RESOURCE_EXHAUSTED" in str(err) else 2 ** (attempt + 1)
            print(f"   ⏳ Transient error ({type(err).__name__}), retrying in {wait_s}s...")
            time.sleep(wait_s)

🚀 Gemini LLM ready: gemini-3.5-flash-lite (temperature=0.3)


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

STATIONERY_CATALOGUE = {
    "spiral notebook (200 pgs)": 85,
    "a4 copier paper ream (500 sheets)": 320,
    "gel pen pack (10 pens, blue/black)": 150,
    "pastel highlighter set (6 colors)": 180,
    "heavy-duty stapler + pins": 240,
    "sticky notes pack (400 sheets)": 95,
    "geometry box set": 160,
    "whiteboard marker set (4 colors)": 120,
    "permanent marker (black)": 35,
    "clear file folder (pack of 5)": 110,
}

CATALOGUE_TEXT = "\n".join(
    f"  - {item}: Rs {price}" for item, price in STATIONERY_CATALOGUE.items()
)

BRIGHTPAGE_SYSTEM_V1 = f"""ROLE:   You are the friendly assistant for BrightPage Stationery & Office Supplies.
FACTS:  Catalogue (INR):
{CATALOGUE_TEXT}
        Hours: Mon-Sat 9:00 AM - 8:30 PM, Sunday 10:00 AM - 4:00 PM.
        Delivery: Free local delivery on orders above Rs 500; Rs 40 fee below Rs 500.
        Bulk discount: 10% off on 10+ identical items or orders above Rs 1000.
        Returns: Damaged items replaced within 3 days with receipt.
RULES:  Never promise delivery outside local city limits. If an item isn't in FACTS,
        say "I'll check with the store manager about availability and get back to you shortly."
        Work out totals carefully.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer.
        Write prices as plain numbers without commas (e.g. Rs 1000)."""

prompt = ChatPromptTemplate.from_messages([
    ("system", BRIGHTPAGE_SYSTEM_V1),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = invoke_retry(chain, {"history": history, "question": question})
    time.sleep(4.2)
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you offer free delivery?"))
print(ask("What's my name?"))

Hi Ravi! Yes, we offer free local delivery on orders above Rs 500. For orders below Rs 500, there is just a Rs 40 delivery fee. Let me know if you'd like to place an order today!


Your name is Ravi! How can I help you with our stationery supplies today?


In [4]:
customer_questions = [
    "What is the delivery fee for a Rs 300 order?",
    "Do you deliver outside the city?",
    "Do you sell scientific calculators?",
    "What is the price of 6 spiral notebooks?",
]

for q in customer_questions:
    print(f"Customer: {q}")
    print(f"Assistant: {ask(q)}\n")

Customer: What is the delivery fee for a Rs 300 order?


Assistant: Hi Ravi! The delivery fee for a Rs 300 order is Rs 40, since free local delivery applies only to orders above Rs 500. Would you like to add anything else to reach the free delivery limit?

Customer: Do you deliver outside the city?


Assistant: Hi Ravi! I'm sorry, but we only offer local delivery and cannot promise delivery outside city limits. Let me know if you need any stationery delivered locally!

Customer: Do you sell scientific calculators?


Assistant: Hi Ravi! I'll check with the store manager about availability and get back to you shortly. Let me know if you need anything else from our catalogue in the meantime!

Customer: What is the price of 6 spiral notebooks?


Assistant: Hi Ravi! Each 200-page spiral notebook is Rs 85, so 6 notebooks total Rs 510. Since your order is above Rs 500, you also qualify for free local delivery!



In [5]:
TESTS_CSV = """question,must_include
How much is a spiral notebook?,85
What is the price of 2 A4 copier paper reams?,640
Are you open on Sunday?,10:00
Is delivery free for an order of Rs 600?,free
What is the delivery fee for a Rs 300 order?,40
Do you sell scientific calculators?,store manager
What is your return policy for damaged items?,3 days
What discount do I get if I buy 10 gel pen packs?,10%
How much is 1 geometry box set and 1 sticky notes pack?,255
Do you deliver outside the city?,city
"""

if not os.path.exists("tests.csv"):
    with open("tests.csv", "w", encoding="utf-8") as f:
        f.write(TESTS_CSV)
    print("✅ Created tests.csv")
else:
    print("ℹ️ Using existing tests.csv")

ℹ️ Using existing tests.csv


In [6]:
import csv
import time

def run_tests(path: str = "tests.csv", test_chain=None, delay: float = 4.2):
    active_chain = test_chain or chain
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    passed = 0
    answers = []
    for row in rows:
        for attempt in range(5):
            try:
                answer = active_chain.invoke({"history": [], "question": row["question"]})
                break
            except (KeyError, ValueError, FileNotFoundError):
                raise
            except Exception as err:
                if attempt == 4:
                    raise
                wait_s = 35 if "429" in str(err) or "RESOURCE_EXHAUSTED" in str(err) else 2 ** (attempt + 1)
                print(f"   ⏳ Transient error ({type(err).__name__}), retrying in {wait_s}s...")
                time.sleep(wait_s)
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        answers.append(answer)
        print("PASS" if ok else "FAIL", "|", row["question"], "->", answer[:70])
        if delay:
            time.sleep(delay)
    rate = passed / len(rows)
    print(f"\n{passed}/{len(rows)} passed ({rate:.0%})")
    return rate, answers

print("=== Baseline Prompt (V1) ===")
v1_rate, v1_answers = run_tests("tests.csv")

=== Baseline Prompt (V1) ===


PASS | How much is a spiral notebook? -> Hello! A spiral notebook (200 pgs) is just Rs 85 at BrightPage Station


PASS | What is the price of 2 A4 copier paper reams? -> Hello! 

Two A4 copier paper reams (500 sheets each) come to Rs 640. 



PASS | Are you open on Sunday? -> Yes, we are open on Sundays from 10:00 AM to 4:00 PM! How can I help y


PASS | Is delivery free for an order of Rs 600? -> Yes, absolutely! Since your order is above Rs 500, local delivery is c


PASS | What is the delivery fee for a Rs 300 order? -> Hello! Since your order total is below Rs 500, the delivery fee for lo


... (2 outputs left out)

PASS | What discount do I get if I buy 10 gel pen packs? -> Hello! You get a 10% bulk discount when you buy 10 or more identical i


PASS | How much is 1 geometry box set and 1 sticky notes pack? -> Hello! A geometry box set is Rs 160 and a sticky notes pack is Rs 95. 


PASS | Do you deliver outside the city? -> Hello! We only offer local delivery within city limits—free for orders



10/10 passed (100%)


In [7]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=BRIGHTPAGE_SYSTEM_V1)   # tools come on Day 4
result = invoke_retry(agent, {"messages": [{"role": "user", "content": "Do you deliver outside the city?"}]})
print(result["messages"][-1].text)

Hello! I'm sorry, but we only offer delivery within our local city limits. We'd love to help you with local delivery or in-store pickup, though! Let me know if you need anything from our stationery catalogue.


### Prompt Iteration (`V1` -> `V2`) & Temperature Stability (`1.0`)

In [8]:
BRIGHTPAGE_SYSTEM_V2 = BRIGHTPAGE_SYSTEM_V1 + """
        Mention 'local city limits' when asked about out-of-city delivery, and answer only what the customer asked."""

bp_prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", BRIGHTPAGE_SYSTEM_V2),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
bp_chain_v2 = bp_prompt_v2 | llm | StrOutputParser()

print("=== Improved Prompt (V2) ===")
v2_rate, v2_answers = run_tests("tests.csv", test_chain=bp_chain_v2)
print(f"\nPass rate change: {v1_rate:.0%} -> {v2_rate:.0%}")

=== Improved Prompt (V2) ===


PASS | How much is a spiral notebook? -> Hello! A 200-page spiral notebook is Rs 85. Let me know if you need an


PASS | What is the price of 2 A4 copier paper reams? -> Hello! Two A4 copier paper reams (500 sheets each) will cost Rs 640. S


PASS | Are you open on Sunday? -> Yes, we are open on Sundays from 10:00 AM to 4:00 PM! Let me know if y


PASS | Is delivery free for an order of Rs 600? -> Yes, delivery is completely free for your order of Rs 600 since it is 


PASS | What is the delivery fee for a Rs 300 order? -> Hello! The delivery fee for a Rs 300 order is Rs 40. We only offer del


... (2 outputs left out)

PASS | What discount do I get if I buy 10 gel pen packs? -> Hello! You get a 10% discount when you buy 10 identical items like the


PASS | How much is 1 geometry box set and 1 sticky notes pack? -> Hello! One geometry box set is Rs 160 and one sticky notes pack is Rs 


PASS | Do you deliver outside the city? -> Hello! We only offer delivery within local city limits.



10/10 passed (100%)

Pass rate change: 100% -> 100%


In [9]:
llm_hot = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=1.0, max_retries=3, timeout=30)
bp_chain_hot = bp_prompt_v2 | llm_hot | StrOutputParser()

print("=== Temperature 1.0 — Run 1 ===")
hot_rate_1, hot_answers_1 = run_tests("tests.csv", test_chain=bp_chain_hot)

print("\n=== Temperature 1.0 — Run 2 ===")
hot_rate_2, hot_answers_2 = run_tests("tests.csv", test_chain=bp_chain_hot)

changed = sum(a != b for a, b in zip(hot_answers_1, hot_answers_2))
print(f"\nChanged answers between Run 1 and Run 2: {changed}/{len(hot_answers_1)}")
print(f"Pass rates: Run 1 = {hot_rate_1:.0%}, Run 2 = {hot_rate_2:.0%}")

=== Temperature 1.0 — Run 1 ===


PASS | How much is a spiral notebook? -> Hello! A spiral notebook (200 pgs) is Rs 85. Let me know if you need a


PASS | What is the price of 2 A4 copier paper reams? -> Hello! Two A4 copier paper reams (500 sheets each) cost Rs 640 in tota


PASS | Are you open on Sunday? -> Yes, we are open on Sundays from 10:00 AM to 4:00 PM! Let me know if y


PASS | Is delivery free for an order of Rs 600? -> Yes! Delivery is completely free for orders above Rs 500 within our lo


PASS | What is the delivery fee for a Rs 300 order? -> Hello! The delivery fee for an order of Rs 300 is Rs 40. We only offer


... (13 outputs left out)

PASS | What discount do I get if I buy 10 gel pen packs? -> You get a 10% bulk discount on 10 identical items! For 10 gel pen pack


PASS | How much is 1 geometry box set and 1 sticky notes pack? -> Hello! One geometry box set is Rs 160 and one sticky notes pack is Rs 


PASS | Do you deliver outside the city? -> Hello! We only deliver within local city limits.



10/10 passed (100%)

Changed answers between Run 1 and Run 2: 9/10
Pass rates: Run 1 = 100%, Run 2 = 100%


In [10]:
business_jobs = [
    (
        "Calculate cart subtotal, apply 10% bulk discount, and add Rs 40 delivery fee",
        "plain code",
        "Deterministic math with fixed business rules needs zero LLM reasoning.",
    ),
    (
        "Classify an incoming WhatsApp message as order, inquiry, or complaint",
        "one LLM call",
        "Single-step natural language classification without external tool calls.",
    ),
    (
        "Check live inventory, compute bulk pricing, and record a confirmed order in the sheet",
        "agent",
        "Multi-step task where the model decides which tools to call based on user input.",
    ),
]

for job, label, reason in business_jobs:
    print(f"[{label.upper()}] {job}\n   Why: {reason}\n")

[PLAIN CODE] Calculate cart subtotal, apply 10% bulk discount, and add Rs 40 delivery fee
   Why: Deterministic math with fixed business rules needs zero LLM reasoning.

[ONE LLM CALL] Classify an incoming WhatsApp message as order, inquiry, or complaint
   Why: Single-step natural language classification without external tool calls.

[AGENT] Check live inventory, compute bulk pricing, and record a confirmed order in the sheet
   Why: Multi-step task where the model decides which tools to call based on user input.

